In [51]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc, sum, avg, round, count,max
spark = SparkSession.builder.appName("DemoSparkApp").getOrCreate()
from pyspark.sql.functions import *
from pyspark.sql.window import Window


In [47]:
customers = spark.read.csv("./Data/customers.csv",
                           header=True,
                           inferSchema=True)

products = spark.read.csv("./Data/products.csv",
                          header=True,
                          inferSchema=True)

orders = spark.read.csv("./Data/orders.csv",
                        header=True,
                        inferSchema=True)

order_items = spark.read.csv("./Data/order_items.csv",
                             header=True,
                             inferSchema=True)

returns = spark.read.csv("./Data/returns.csv",
                         header=True,
                         inferSchema=True)

In [4]:
# Q1
print("customer count:", customers.count())
print("products count:", products.count())
print("orders count:", orders.count())
print("returened order count:", returns.count())

customer count: 100000
products count: 50000
orders count: 1000000
returened order count: 100000


In [5]:
order_items = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Data/order_items.csv")

products = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("Data/products.csv")

26/06/16 03:45:47 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [6]:
order_items.printSchema()
order_items.show(5)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: double (nullable = true)

+-------------+--------+----------+--------+-------------+
|order_item_id|order_id|product_id|quantity|selling_price|
+-------------+--------+----------+--------+-------------+
|            1|  227444|     28849|       5|       727.98|
|            2|   32708|     25471|       2|       203.02|
|            3|  426242|      2818|       5|      1061.81|
|            4|  236965|     35931|       5|      1005.49|
|            5|  289552|     48310|       4|       540.78|
+-------------+--------+----------+--------+-------------+
only showing top 5 rows



In [7]:
# Q2
from pyspark.sql.functions import col, sum as _sum

sales_by_category = (
    order_items
    .join(products, "product_id")
    .groupBy("category")
    .agg(
        _sum(col("quantity") * col("selling_price")).alias("total_sales")
    )
)

sales_by_category.show()

[Stage 28:=======>                                                  (1 + 7) / 8]

+--------------+-------------------+
|      category|        total_sales|
+--------------+-------------------+
|Home & Kitchen|7.581388732800013E8|
|        Sports|7.433388681299993E8|
|   Electronics|7.442665041099994E8|
|      Clothing|7.419227945700003E8|
|         Books|7.464907783500001E8|
|        Beauty|7.626693058999988E8|
|          Toys|7.446190722999973E8|
+--------------+-------------------+



In [8]:
# Q3
top_customer=(
    orders
    .join(order_items,"order_id")
    .groupBy("customer_id")
    .agg(
        sum(col("selling_price")).alias("total_purchase")
    )
    .orderBy(desc("total_purchase"))
    .limit(10)
)

top_customer.show()

[Stage 35:====================================>                     (5 + 3) / 8]

+-----------+------------------+
|customer_id|    total_purchase|
+-----------+------------------+
|      23289|56384.689999999995|
|      93094|          56147.91|
|      52275|51738.220000000016|
|      65572|50453.869999999995|
|      20648|50164.020000000004|
|      61218|50117.229999999996|
|      42327|          49874.78|
|      87660|49141.329999999994|
|      40442|          48800.07|
|      82699|          48530.14|
+-----------+------------------+



In [9]:
# Q4
from pyspark.sql.functions import col, to_date, year, month, max as _max, sum as _sum, desc

last_year= orders.select(year(_max("order_date"))).collect()[0][0]
print(last_year)
df = (
    orders
    .join(order_items,"order_id")
    .withColumn("order_year",year("order_date"))
    .withColumn("order_month", month("order_date"))
    .filter(col("order_year")==last_year)
)

monthly_sales = (
    df.groupBy("order_month")
    .agg(
        sum(col("quantity")*col("selling_price")).alias("total_sales")
    )
    .orderBy("order_month")
)

monthly_sales.show()

2024


[Stage 44:==============>                                           (2 + 6) / 8]

+-----------+--------------------+
|order_month|         total_sales|
+-----------+--------------------+
|          1| 4.445777757599986E8|
|          2|4.1536614420000046E8|
|          3| 4.436282454100011E8|
|          4|4.2782097434000105E8|
|          5| 4.448106189500004E8|
|          6| 4.317051540599995E8|
|          7| 4.436705191199995E8|
|          8| 4.410951770200002E8|
|          9| 4.310715260800007E8|
|         10| 4.413637893099995E8|
|         11| 4.336233640399993E8|
|         12|4.4271290835000044E8|
+-----------+--------------------+



In [23]:
# Q5 find the return percentage for each product category
# order_items.printSchema()
# products.printSchema()
# returns.printSchema()

total_sold_quantity = (
    order_items
    .join(products,"product_id")
    .groupBy("category")
    .agg(
        sum(col("quantity")).alias("total_units")
    )
)

total_sold_quantity.show()

returned_quantity = (
    order_items
    .join(returns,"order_id","inner")
    .join(products,"product_id","inner")
    .groupBy("category")
    .agg(
        sum("quantity").alias("reutned_qty")
    )
)
returned_quantity.show()

return_total = (
    total_sold_quantity
    .join(returned_quantity,"category")
)
return_total.show()

returned_percentage = (
    return_total
    .withColumn(
        "return_percentage",
        round(
            (col("reutned_qty")/col("total_units"))*100,
            2
        )
    )
)
returned_percentage.show()

+--------------+-----------+
|      category|total_units|
+--------------+-----------+
|Home & Kitchen|    1302863|
|        Sports|    1272271|
|   Electronics|    1276881|
|      Clothing|    1281718|
|         Books|    1281139|
|        Beauty|    1292321|
|          Toys|    1292083|
+--------------+-----------+



+--------------+-----------+
|      category|reutned_qty|
+--------------+-----------+
|Home & Kitchen|     130221|
|        Sports|     127587|
|   Electronics|     127917|
|      Clothing|     128215|
|         Books|     128863|
|        Beauty|     129583|
|          Toys|     130349|
+--------------+-----------+



+--------------+-----------+-----------+
|      category|total_units|reutned_qty|
+--------------+-----------+-----------+
|Home & Kitchen|    1302863|     130221|
|        Sports|    1272271|     127587|
|   Electronics|    1276881|     127917|
|      Clothing|    1281718|     128215|
|         Books|    1281139|     128863|
|        Beauty|    1292321|     129583|
|          Toys|    1292083|     130349|
+--------------+-----------+-----------+



+--------------+-----------+-----------+-----------------+
|      category|total_units|reutned_qty|return_percentage|
+--------------+-----------+-----------+-----------------+
|Home & Kitchen|    1302863|     130221|             9.99|
|        Sports|    1272271|     127587|            10.03|
|   Electronics|    1276881|     127917|            10.02|
|      Clothing|    1281718|     128215|             10.0|
|         Books|    1281139|     128863|            10.06|
|        Beauty|    1292321|     129583|            10.03|
|          Toys|    1292083|     130349|            10.09|
+--------------+-----------+-----------+-----------------+



In [44]:
# Q6 Determine the most preferred payment mode in each state
# orders.printSchema()
# customers.printSchema()

from pyspark.sql.functions import max as _max
df = (
    customers
    .join(orders,"customer_id")
)

payment_counts = (
    df.groupBy("state","payment_mode")
    .agg(
        count("*").alias("order_count"))
)
payment_counts.show()

max_counts = (
    payment_counts
    .groupBy("state")
    .agg(_max("order_count").alias("max_order_count"))
)
max_counts.show()

result = (
    payment_counts
    .join(max_counts,"state")
    .filter(payment_counts.order_count == max_counts.max_order_count)
    .select("state","payment_mode","order_count")
)

result.show()

+-----+----------------+-----------+
|state|    payment_mode|order_count|
+-----+----------------+-----------+
|   FL|     Net Banking|      19629|
|   NC|     Net Banking|      19596|
|   GA|      Debit Card|      19733|
|   OH|             UPI|      19930|
|   MI|Cash on Delivery|      20205|
|   IL|Cash on Delivery|      20498|
|   OH|     Net Banking|      20351|
|   TX|     Credit Card|      19874|
|   IL|             UPI|      20359|
|   NY|             UPI|      20108|
|   NY|     Net Banking|      20229|
|   TX|             UPI|      20065|
|   GA|     Credit Card|      19722|
|   TX|      Debit Card|      19988|
|   MI|     Net Banking|      20116|
|   NY|Cash on Delivery|      20208|
|   CA|      Debit Card|      20024|
|   IL|     Net Banking|      20404|
|   CA|     Credit Card|      19979|
|   OH|     Credit Card|      19794|
+-----+----------------+-----------+
only showing top 20 rows

+-----+---------------+
|state|max_order_count|
+-----+---------------+
|   MI|       

In [48]:
# Q7 identify customers who have purchases products from at least 5 different categories and spen more then 1,00,000.

customer_analysis = (
    orders
    .join(order_items,"order_id")
    .join(products,"product_id")
    .join(customers,"customer_id")
    .withColumn(
        "amount",
        col("quantity")*col("selling_price")
    )
    .groupBy(
        "customer_id","customer_name")
    .agg(
        countDistinct("category").alias("categories_bought"),
        round(sum("amount"),2).alias("total_spend")
    )
    .filter(
        (col("categories_bought")>=5)&
        (col("total_spend")>100000)
    )
)

customer_analysis.show()

26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/06/16 04:53:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
[Stage 250:=======>                                             

+-----------+--------------+-----------------+-----------+
|customer_id| customer_name|categories_bought|total_spend|
+-----------+--------------+-----------------+-----------+
|      84144|Customer_84144|                7|  103326.02|
|      46860|Customer_46860|                7|  100298.43|
|      41157|Customer_41157|                7|   105187.1|
|      52297|Customer_52297|                7|  107812.68|
|      26241|Customer_26241|                7|   121047.8|
|      97920|Customer_97920|                7|  117591.17|
|      90203|Customer_90203|                7|  109729.63|
|      18149|Customer_18149|                7|   101780.7|
|      46060|Customer_46060|                7|  115805.04|
|      60620|Customer_60620|                7|  110953.51|
|      28078|Customer_28078|                7|  115138.03|
|      67631|Customer_67631|                7|  106734.26|
|      68399|Customer_68399|                7|   119426.7|
|      53139|Customer_53139|                7|  127296.5

In [55]:
# Q8 find the top 3 products by revenue within each category

product_revenue = (
    order_items
    .join(products,"product_id")
    .groupBy("category","product_id","product_name")
    .agg(
        sum(
            col("quantity")*col("selling_price")
        ).alias("revenue")
    )
)
product_revenue.show()

windows_spec = (
    Window
    .partitionBy("category")
    .orderBy(desc("revenue"))
)


top_products = (
    product_revenue
    .withColumn(
        "rank",dense_rank().over(windows_spec)
    )
    .filter(col("rank")<=3)
    .orderBy("category","rank")
)

top_products.show()

+-----------+----------+-------------+------------------+
|   category|product_id| product_name|           revenue|
+-----------+----------+-------------+------------------+
|   Clothing|      1280| Product_1280|         185289.69|
|Electronics|     24507|Product_24507| 90113.40000000001|
|Electronics|     37498|Product_37498|          53721.08|
|     Beauty|     18640|Product_18640|         107917.53|
|     Sports|     45205|Product_45205|114789.15999999999|
|     Beauty|      1893| Product_1893|          18459.02|
|Electronics|     17156|Product_17156|         158318.22|
|       Toys|     44426|Product_44426|101340.68999999999|
|Electronics|     21030|Product_21030|105387.98999999999|
|Electronics|     49242|Product_49242|          72795.13|
|     Sports|     11448|Product_11448| 74467.65999999999|
|     Sports|      1300| Product_1300|         164779.81|
|       Toys|     37099|Product_37099|          83755.54|
|   Clothing|     15493|Product_15493|         101753.73|
|Electronics| 

[Stage 273:==============>                                          (2 + 6) / 8]

+--------------+----------+-------------+------------------+----+
|      category|product_id| product_name|           revenue|rank|
+--------------+----------+-------------+------------------+----+
|        Beauty|     44016|Product_44016|         277567.99|   1|
|        Beauty|     14849|Product_14849|          274894.2|   2|
|        Beauty|       786|  Product_786|272174.69999999995|   3|
|         Books|     35314|Product_35314|296468.77999999997|   1|
|         Books|     28311|Product_28311|286757.72000000003|   2|
|         Books|     37479|Product_37479|         276736.71|   3|
|      Clothing|      7025| Product_7025|293821.97000000003|   1|
|      Clothing|      1560| Product_1560|288474.08999999997|   2|
|      Clothing|     31322|Product_31322|         282241.17|   3|
|   Electronics|      6719| Product_6719|         299113.87|   1|
|   Electronics|     23519|Product_23519|289561.72000000003|   2|
|   Electronics|     38170|Product_38170|288875.23000000004|   3|
|Home & Ki